# Capítulo 8 · Tareas largas y fiabilidad

Un cliente de la Clínica Veterinaria Naranco ve dos cargos de la misma factura. Se reproduce el fallo con una pasarela de pago que a veces tarda más de lo que el cliente espera, y se prueban cuatro políticas de reintentos en el anfitrión, sin modelo, y cuatro anfitriones con el agente: con y sin claves de idempotencia, con y sin una forma de consultar el estado del cobro.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. La primera parte no la necesita.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar los datos y la pasarela

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 8: tareas largas y fiabilidad. Un cliente ve dos cargos de la misma factura:
tiempos de espera, reintentos y claves de idempotencia."""
import asyncio
import hashlib
import json
import re
import time
import sys
import urllib.request
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client, MCPError
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
MODELO = None                               # None: elegir según el entorno
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/"
HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def descargar(archivo):
    """Un archivo JSON de la carpeta datos; en Colab, se descarga."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return json.loads(ruta.read_text(encoding="utf-8"))


def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar nombres escritos de cualquier forma."""
    return str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu")).strip()


def cargar_datos():
    global DUENOS
    DUENOS = descargar("duenos.json")

In [ ]:
ESPERA = 1.0                      # segundos que espera el cliente cada respuesta


LENTO = 1.5                       # lo que tarda el banco las veces que va lento


def cargar_cobros():
    """Las facturas sin pagar de la clínica, con un perfil de velocidad para su primer cobro."""
    global FACTURAS, PENDIENTES, PERFILES
    FACTURAS = {f["id"]: f for f in descargar("facturas.json")}
    PENDIENTES = [i for i, f in FACTURAS.items() if not f["pagada"]]
    ciclo = ["rápido", "cobra y tarda", "rápido", "tarda y cobra"]
    PERFILES = {i: ciclo[n % len(ciclo)] for n, i in enumerate(PENDIENTES)}
    reiniciar_cobros()


def reiniciar_cobros():
    """COBROS son los cargos que ha hecho el banco; CONFIRMADAS, las facturas cuyo cobro ha
    llegado a confirmarse a la clínica (las que su programa de facturación da por pagadas)."""
    global COBROS, CONFIRMADAS, INTENTOS, CLAVES
    COBROS, CONFIRMADAS, INTENTOS, CLAVES = [], set(), {}, {}


async def banco(factura, perfil):
    """El banco: cobra el importe de la factura, antes o después de tardar."""
    if perfil == "tarda y cobra":
        await asyncio.sleep(LENTO)
    COBROS.append(factura)
    await asyncio.sleep(LENTO if perfil == "cobra y tarda" else 0.05)
    return {"cobrada": factura, "importe": FACTURAS[factura]["total"]}


def crear_pasarela(idempotencia=False, con_estado=True):
    """El servidor de cobros. Con idempotencia, cobrar_factura acepta una clave: si ya ha
    recibido esa clave, no cobra otra vez, sino que devuelve el resultado del primer cobro.
    Con con_estado, ofrece también estado_factura, que pregunta al banco si se ha cobrado."""
    servidor = MCPServer("cobros")

    @servidor.tool()
    async def cobrar_factura(factura: Annotated[str, Field(description="Id de la factura, como F-0024.")],
                             clave: Annotated[str, Field(description="Clave de idempotencia: la misma "
                                                         "en los reintentos de un mismo cobro.")] = "") -> dict:
        """Cobra una factura con la tarjeta del cliente."""
        if factura not in FACTURAS:
            raise ToolError(f"No existe la factura {factura}.")
        if idempotencia and clave in CLAVES:
            resultado = await asyncio.shield(CLAVES[clave])  # el mismo cobro, no uno nuevo
            CONFIRMADAS.add(factura)
            return resultado
        INTENTOS[factura] = INTENTOS.get(factura, 0) + 1
        perfil = PERFILES.get(factura, "rápido") if INTENTOS[factura] == 1 else "rápido"
        orden = asyncio.ensure_future(banco(factura, perfil))  # la orden sale hacia el banco
        if idempotencia and clave:
            CLAVES[clave] = orden
        resultado = await asyncio.shield(orden)               # esperar no es lo mismo que cobrar
        CONFIRMADAS.add(factura)
        return resultado

    if con_estado:
        @servidor.tool()
        def estado_factura(factura: Annotated[str, Field(description="Id de la factura.")]) -> dict:
            """Pregunta al banco si una factura ya se ha cobrado."""
            return {"factura": factura, "cobrada": factura in COBROS}

    @servidor.tool()
    def facturas_pendientes(dueno: Annotated[str, Field(description="Nombre del cliente.")]) -> dict:
        """Facturas sin cobrar de un cliente, con su importe."""
        ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
        return {"facturas": [{"id": i, "total": f["total"]} for i, f in FACTURAS.items()
                             if f["dueno"] in ids and not f["pagada"] and i not in CONFIRMADAS]}

    return servidor

In [ ]:
cargar_datos()
cargar_cobros()
print(PERFILES)

## 2. Lo que pasa cuando se agota el tiempo

El cliente deja de esperar al segundo, pero el banco ya ha recibido la orden.

In [ ]:
async def probar():
    reiniciar_cobros()
    async with Client(crear_pasarela(), read_timeout_seconds=ESPERA) as c:
        try:
            await c.call_tool("cobrar_factura", {"factura": "F-0014"})
        except MCPError as e:
            print("El cliente:", e)
    await asyncio.sleep(LENTO)
    print("El banco ha cobrado:", COBROS)

await probar()

## 3. Cuatro políticas de reintentos, sin modelo

In [ ]:
async def cobrar_con(cliente, factura, politica):
    """Un cobro con una política de reintentos. Devuelve si el anfitrión lo da por hecho."""
    clave = f"cobro-{factura}" if politica == "clave de idempotencia" else ""
    for intento in range(3):
        try:
            r = await cliente.call_tool("cobrar_factura", {"factura": factura, "clave": clave})
            return not r.is_error
        except MCPError:                                  # tiempo agotado
            if politica == "sin reintentos":
                return False
            if politica == "comprobar antes de reintentar":
                estado = await cliente.call_tool("estado_factura", {"factura": factura})
                if json.loads(estado.content[0].text)["cobrada"]:
                    return True
    return False


POLITICAS = ["sin reintentos", "reintentar", "comprobar antes de reintentar", "clave de idempotencia"]


async def cobrar_todas(politica):
    reiniciar_cobros()
    inicio = time.perf_counter()
    async with Client(crear_pasarela(idempotencia=True), read_timeout_seconds=ESPERA) as c:
        hechas = {f: await cobrar_con(c, f, politica) for f in PENDIENTES}
    await asyncio.sleep(LENTO + 0.2)                     # las órdenes que aún estén en camino
    veces = {f: COBROS.count(f) for f in PENDIENTES}
    return {"cobros_duplicados": sum(max(0, n - 1) for n in veces.values()),
            "cobradas": sum(n > 0 for n in veces.values()),
            "cobradas_pero_dadas_por_fallidas": sum(veces[f] > 0 and not hechas[f] for f in PENDIENTES),
            "segundos": round(time.perf_counter() - inicio, 1)}

In [ ]:
for politica in POLITICAS:
    print(f"{politica:32s}", await cobrar_todas(politica))

## 4. El agente cobra

Si la respuesta dice que no se ha cobrado, la recepcionista repite la petición.

In [ ]:
tok, modelo = cargar(MODELO)

In [ ]:
SISTEMA = (
    "You are the assistant of the staff of Clínica Veterinaria Naranco, a veterinary "
    "clinic in Oviedo (Spain). Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in Spanish, in one or two sentences.")


def generar(mensajes, esquemas, max_tokens=400):
    """Una respuesta del modelo y los tokens de entrada y de salida."""
    texto = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False,
                                    add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    return (tok.decode(nuevos, skip_special_tokens=True).strip(),
            entrada["input_ids"].shape[1], len(nuevos))


def leer_llamadas(respuesta):
    """Las llamadas que pide el modelo, escritas entre <tool_call> y </tool_call>."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)</tool_call>", respuesta, re.S):
        try:
            llamada = json.loads(bloque)
            llamadas.append({"name": llamada["name"],
                             "arguments": llamada.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def dice(respuesta, *formas):
    """¿Aparece en la respuesta alguna de esas formas?"""
    return any(re.search(f, respuesta or "", re.I) for f in formas)


def sin_titulos(esquema):
    """El esquema sin las claves «title» que añade Pydantic y que el modelo no necesita."""
    if isinstance(esquema, dict):
        return {k: sin_titulos(v) for k, v in esquema.items() if k != "title"}
    return esquema


def compactar(texto):
    """El JSON de un resultado en una sola línea, sin la sangría con que lo envía el servidor."""
    try:
        return json.dumps(json.loads(texto), ensure_ascii=False)
    except ValueError:
        return texto


async def agente_mcp(tarea, anfitrion, max_pasos=8, ver=False):
    """El bucle del capítulo 1, con las herramientas de los servidores MCP."""
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": tarea}]
    traza = []
    for paso in range(1, max_pasos + 1):
        respuesta, entrada, salida = generar(mensajes, anfitrion.esquemas)
        llamadas = leer_llamadas(respuesta)[:1]
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        traza.append({"paso": paso, "tokens_entrada": entrada, "pensamiento": pensamiento,
                      "llamadas": []})
        if not llamadas:
            if ver:
                print(f"[{paso}] Respuesta: {respuesta}")
            return respuesta, traza
        mensajes.append({"role": "assistant", "content": pensamiento, "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            texto = await anfitrion.llamar(llamada["name"], llamada["arguments"])
            traza[-1]["llamadas"].append({**llamada, "resultado": texto,
                                          "tokens_resultado": len(tok(texto)["input_ids"])})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?", "content": texto})
            if ver:
                print(f"[{paso}] {llamada['name']}({json.dumps(llamada['arguments'], ensure_ascii=False)})")
                print(f"     → {texto[:150]}")
    return None, traza

In [ ]:
class Anfitrion:
    """El anfitrión compacto del capítulo 4, para la pasarela. Si una llamada agota el tiempo,
    se lo dice al modelo con un error. Con claves, añade a cada cobro una clave de
    idempotencia derivada de la tarea y de los argumentos: si el modelo reintenta el mismo
    cobro, la clave es la misma y la pasarela no cobra dos veces. Con explicar, el error del
    tiempo agotado dice además qué hacer: no repetir el cobro y consultar su estado."""

    def __init__(self, servidor, claves=False, explicar=False):
        self.servidor, self.claves, self.explicar, self.tarea = servidor, claves, explicar, ""

    async def __aenter__(self):
        self.cliente = Client(self.servidor, read_timeout_seconds=ESPERA)
        await self.cliente.__aenter__()
        self.esquemas = []
        for h in (await self.cliente.list_tools()).tools:
            parametros = sin_titulos(h.input_schema)
            parametros["properties"].pop("clave", None)        # la clave la pone el anfitrión
            self.esquemas.append({"type": "function", "function": {
                "name": h.name, "description": h.description, "parameters": parametros}})
        return self

    async def __aexit__(self, *error):
        await self.cliente.__aexit__(*error)

    async def llamar(self, nombre, argumentos):
        if nombre == "cobrar_factura" and self.claves:
            huella = json.dumps([self.tarea, argumentos], sort_keys=True, ensure_ascii=False)
            argumentos = {**argumentos, "clave": hashlib.sha256(huella.encode()).hexdigest()[:16]}
        try:
            r = await self.cliente.call_tool(nombre, argumentos)
        except MCPError:
            error = f"La pasarela no ha respondido en {ESPERA:g} s; no se sabe si la operación se ha hecho."
            if self.explicar:
                error += (" Puede que el cobro sí se haya hecho: no lo repitas. Consulta estado_factura "
                          "para saberlo antes de responder.")
            return json.dumps({"error": error}, ensure_ascii=False)
        return compactar("\n".join(b.text for b in r.content if b.type == "text"))


def tareas_de_cobro():
    """Una tarea por cada factura pendiente de los seis primeros clientes."""
    tareas = []
    for f in PENDIENTES[:6]:
        dueno = DUENOS[FACTURAS[f]["dueno"]]["nombre"]
        tareas.append((f"cobrar_{f}", f"Cobra la factura {f} a {dueno}.", f))
    return tareas


def dice_cobrada(respuesta):
    """¿Dice la respuesta que la factura se ha cobrado?"""
    if dice(respuesta, r"ya (ha|había|habia) sido (cobrad|pagad)", r"ya (fue|está|estaba) (cobrad|pagad)"):
        return True
    return dice(respuesta, "cobrad", "se ha cobrado", "pagad", "realizad") and not dice(
        respuesta, r"no (se )?(ha|han|hemos|he|pudo|puede|se pudo|se ha podido)", "no sabemos",
        "no se sabe", "no está claro", "no hay confirmación", "no se ha confirmado")


async def agente_cobra(claves, explicar, con_estado):
    """Las seis tareas de cobro con el agente. Si la respuesta dice que no se ha cobrado, la
    recepcionista repite la misma petición, como haría cualquiera. Se cuentan los cobros
    duplicados y si la última respuesta dice bien lo que ha pasado."""
    filas = []
    for clave, tarea, factura in tareas_de_cobro():
        reiniciar_cobros()
        respuestas, llamadas = [], []
        async with Anfitrion(crear_pasarela(True, con_estado), claves, explicar) as anfitrion:
            anfitrion.tarea = tarea
            for _ in range(2):
                respuesta, traza = await agente_mcp(tarea, anfitrion)
                respuestas.append(respuesta)
                llamadas += [(l["name"], l["arguments"]) for p in traza for l in p["llamadas"]]
                await asyncio.sleep(LENTO + 0.2)          # las órdenes que aún estén en camino
                if dice_cobrada(respuesta):
                    break
        veces = COBROS.count(factura)
        filas.append({"tarea": clave, "perfil": PERFILES[factura], "cobros": veces,
                      "peticiones": len(respuestas),
                      "respuesta_correcta": dice_cobrada(respuestas[-1]) == (veces > 0),
                      "respuestas": respuestas, "llamadas": llamadas})
    return {"cobros_duplicados": sum(max(0, f["cobros"] - 1) for f in filas),
            "peticiones_repetidas": sum(f["peticiones"] - 1 for f in filas),
            "respuestas_correctas": sum(f["respuesta_correcta"] for f in filas), "filas": filas}


VARIANTES = {"sin claves, sin consulta de estado": (False, False, False),
             "con claves, sin consulta de estado": (True, False, False),
             "sin claves, con consulta de estado": (False, False, True),
             "con claves y un error que explica": (True, True, True)}

In [ ]:
for nombre, (claves, explicar, con_estado) in VARIANTES.items():
    r = await agente_cobra(claves, explicar, con_estado)
    print(f"{nombre:36s}", {k: v for k, v in r.items() if k != "filas"})

## Ejercicios

1. Explica con tus palabras por qué un tiempo de espera agotado no significa que la operación haya fallado.

2. Sube el tiempo de espera a 2 segundos y repite las cuatro políticas. ¿Desaparecen los duplicados? ¿Qué pasaría si el banco tardara 3?

3. Cambia los perfiles para que la pasarela tarde siempre en cobrar («tarda y cobra»). ¿Cuántos duplicados da la política que comprueba antes de reintentar?

4. Con la clave de idempotencia, ¿qué pasa si los cobros de dos facturas distintas llegan con la misma clave? ¿Cómo debería defenderse la pasarela?

5. La clave del anfitrión se deriva de la tarea y de los argumentos. Cambia una palabra de la petición repetida de la recepcionista. ¿Sigue protegiendo la clave?

6. Haz que `cobrar_factura` informe del progreso con `ctx.report_progress` mientras espera al banco, y que el cliente lo muestre con `progress_callback`. ¿Cambia algo en los duplicados?

7. Escribe una política que, tras un tiempo agotado, espere un poco más antes de comprobar el estado. ¿Cuánto hay que esperar para no duplicar nada?

8. Añade a la pasarela un límite de tres cobros por minuto y haz que el anfitrión reintente con una espera creciente. ¿Cuántos cobros salen y cuánto tardan?

9. ¿Qué herramientas de la clínica necesitarían clave de idempotencia? Clasifica las de los capítulos anteriores en consultas y acciones, y di en cuáles un reintento haría daño.

10. **Proyecto.** Añade claves de idempotencia a una herramienta que actúe de tu proyecto del capítulo 4, y escribe una prueba del servidor (capítulo 5) que demuestre que un reintento con la misma clave no repite la acción.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Especificación de MCP](https://modelcontextprotocol.io/specification): cancelación, progreso y tiempos de espera. [SDK de Python](https://github.com/modelcontextprotocol/python-sdk). Lecciones 13.13 y 13.29 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La Clínica Veterinaria Naranco y sus datos son ficticios. Código y experimentos propios.